# ATML PA3 — free Kaggle continuation
Private notebook. Exact Task1 trainer, seed6304, microbatch2 × accumulation16. Attach pa3-task1-input.zip as a **private** Kaggle input. Preserve all /kaggle/working outputs before ending a session. Do not start training while the existing Modal worker remains active. Two T4 devices are separate16GiB devices; this launcher uses one.


In [ ]:
import subprocess, sys, pathlib, json, zipfile, urllib.request
REPO_COMMIT = "e07b5e1548065f5c913a25756d3abd3e421e6666"
work = pathlib.Path("/kaggle/working")
source = work / "pa3-source"
if not source.exists():
    archive = work / "pa3-source.zip"
    urllib.request.urlretrieve(f"https://github.com/sohauv/atml-pa3/archive/{REPO_COMMIT}.zip", archive)
    with zipfile.ZipFile(archive) as z:
        z.extractall(work)
    (work / f"atml-pa3-{REPO_COMMIT}").rename(source)
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "torch==2.6.0", "transformers==4.51.3", "accelerate==1.6.0", "datasets==3.5.0", "numpy==2.2.5"])
print("Pinned source prepared:", REPO_COMMIT)


In [ ]:
# CPU-only input verification; no training starts here.
import sys, pathlib
sys.path.insert(0, str(source))
from scripts.kaggle_task1 import unpack_inputs
archives = list(pathlib.Path("/kaggle/input").rglob("pa3-task1-input.zip"))
if len(archives) != 1:
    raise RuntimeError(f"Attach exactly one private pa3-task1-input.zip; found {len(archives)}")
input_archive = archives[0]
data_dir = unpack_inputs(input_archive, work / "pa3/data")
print("Original training data checksum verified:", data_dir)


In [ ]:
# Enable only after the saved Modal controller/child are terminal with zero tasks.
MODAL_HANDOFF_VERIFIED = False
if not MODAL_HANDOFF_VERIFIED:
    raise RuntimeError("Waiting for verified Modal stopping point. No training launched.")
command = [sys.executable, "-m", "scripts.kaggle_task1", "--archive", str(input_archive),
           "--method", "lora4", "--run-id", "lora4-kaggle-20261030", "--seconds", "18000",
           "--handoff-verified"]
subprocess.check_call(command, cwd=source)


Save a notebook version with outputs after the bounded run finishes. Download the complete run directory before ending the session. Recovery checkpoints include optimizer/scaler/RNG and exact data/config hashes. A partial run is not assignment completion. Resume uses the same run directory and a complete checkpoint; never restart or overwrite existing checkpoints.
